# Step 1, session A of 2 (~9.5 h on a T4)

Advisor's control experiment. Runs: interpolation baseline; smoothness-weight tuning (λ ∈ {0, 0.3, 1, 3}) for MLP, GRU and Mamba on validation frames 8 and 16 (flows 0–1); MLP and GRU final runs (5 flows × 3 seeds, tuned λ, size matched to Mamba); fully observed ceilings. **At the end it prints `LAMBDA_MAMBA` for session B.** Settings: Accelerator GPU T4, Internet on. Run all.

In [ ]:
REPO_URL = "https://github.com/muhammadhussainqureshi/thesis-gs"
CONFIG = "thesis1_step1_a"
BUDGET_HOURS = 11.3
LAMBDA_MAMBA = None    # not used in session A

import os, subprocess, sys, glob, json, shutil, torch
WORK = "/kaggle/working"
CODE = "/kaggle/working/thesis-gs"
DATA = f"{WORK}/data"
OUT = f"{WORK}/{CONFIG}"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Accelerator to GPU")
if os.path.isdir(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
assert os.path.isfile(f"{CODE}/configs/{CONFIG}.json"), "config not found - is the latest code pushed?"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lmdb", "tifffile"], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
cfg = json.load(open(f"{CODE}/configs/{CONFIG}.json"))
for r in cfg["runs"]:
    r["args"] = [str(LAMBDA_MAMBA) if x == "LAMBDA_MAMBA" else x for x in r["args"]]
cfg_path = f"{WORK}/{CONFIG}_used.json"
json.dump(cfg, open(cfg_path, "w"), indent=1)
cmd = [sys.executable, f"{CODE}/bin/run_experiments.py", cfg_path, "--out", OUT, "--data_dir", DATA,
       "--budget_hours", str(BUDGET_HOURS), "--zip", f"{WORK}/{CONFIG}_results.zip"]
subprocess.run(cmd)        # failures are logged per run; the notebook keeps going


In [ ]:
import pandas as pd
f = f"{OUT}/tune_mamba/ns4d/results.csv"
if os.path.exists(f):
    v = pd.read_csv(f).groupby("temporal_reg")["val_frames_psnr"].mean()
    print(v.round(3).to_string())
    print(f"\n>>> For session B, set  LAMBDA_MAMBA = {v.idxmax()}")
else:
    print("tune_mamba did not finish - see the logs")


In [ ]:
from IPython.display import Image, Markdown, display
subprocess.run([sys.executable, f"{CODE}/bin/ns4d_step1_report.py", OUT, "--out", f"{OUT}/step1_report"])
for f in ["per_frame.png", "spectra.png"]:
    if os.path.exists(f"{OUT}/step1_report/{f}"):
        display(Image(filename=f"{OUT}/step1_report/{f}"))
for f in glob.glob(f"{OUT}/**/*.pt", recursive=True):   # keep the zip small
    os.remove(f)
shutil.make_archive(f"{WORK}/{CONFIG}_results", "zip", OUT)
print(f"Download {CONFIG}_results.zip from the Output tab.")
